# HTML and CSS extraction support

**Date:** 2026-08-30  
**Status:** Approved design; written specification pending user review  
**Crate:** `crates/spur-graph`  
**Design epic:** `bd-rnwv`  
**Plan ID:** `a5b85dd2-54c8-417c-bafd-eb480beb86a2`

This notebook is the authoritative design for adding standalone, semantic HTML and CSS extraction to SPUR's tree-sitter graph pipeline. It records the approved scope, graph contracts, integration boundaries, failure behavior, and release gates. Implementation remains blocked until this written specification is reviewed and approved.

## Context and evidence

SPUR's extractor is registry-driven. A supported language consists of:

1. a `Language` variant and path matcher;
2. a tree-sitter grammar and `LanguageConfig`;
3. SPUR-owned tags and relation queries;
4. capture-to-`NodeKind` / `RelationKind` mappings;
5. query-contract, integration, and coverage tests.

The generic pipeline already handles discovery, parsing, captures, lexical containment, definitions, pending edges, and resolution. HTML and CSS therefore do not need specialized extractors.

Dependency seam:

- workspace parser runtime: `tree-sitter 0.25.10`;
- prospective HTML grammar: `tree-sitter-html 0.23.2`;
- prospective CSS grammar: `tree-sitter-css 0.25.0`;
- both grammar crates expose `LANGUAGE.into()`, matching the current adapter shape;
- neither grammar ships `tags.scm`, so SPUR owns the extraction queries.

The solver rule catalog has no parser/language-support family. Formal obligations in this notebook concern the actual finite routing and release policies, not invented parser constraints.

## Goals and non-goals

### Goals

- Route `.html`, `.htm`, and `.css` files through distinct HTML/CSS grammar configurations.
- Emit a low-noise set of durable navigation symbols and dependency/link edges.
- Reuse existing `NodeKind`, `RelationKind`, stable-ID, containment, and resolver behavior.
- Preserve all existing language routing and extraction behavior.
- Make coverage explicit in the query README and executable gate contracts.
- Fail closed when a grammar or query cannot be configured.

### Non-goals

- No SCSS, Sass, Less, XML, Vue, Svelte, JSX, or template-language extensions.
- No recursive parsing of HTML `<script>` or `<style>` raw text.
- No inline `style` attribute parsing.
- No class attribute or ordinary HTML element nodes.
- No ordinary CSS declaration nodes.
- No new graph node/relation kinds, schema version, or artifact version.
- No semantic browser DOM, cascade, inheritance, specificity, or asset fetching model.

## Approved decisions and alternatives

The selected approach is **standalone semantic support**.

Rejected alternatives:

- **Detection-only support:** lower implementation cost, but files would contribute little useful graph information.
- **Full web-document support:** richer cross-language output, but requires a generalized embedded-language parser, source-range translation, identity rules, and additional error boundaries.
- **Exhaustive syntax graph:** emitting every tag, class, and declaration would create high-cardinality noise and conflict with the ontology maturity policy.

The approved design uses the current ontology as a semantic compression layer: only named, navigable regions and dependencies become graph facts.

## Architecture and data flow

`Cargo.toml` and `Cargo.lock` add the exact grammar dependencies. `languages.rs` adds variants, matchers, labels, configs, mappings, registry rows, and gate-contract expectations. `tree_sitter.rs` only extends exhaustive language routing; it does not gain an HTML/CSS-specific extraction branch.

New query sources:

- `crates/spur-graph/queries/html/tags.scm`
- `crates/spur-graph/queries/html/spur-edges.scm`
- `crates/spur-graph/queries/css/tags.scm`
- `crates/spur-graph/queries/css/spur-edges.scm`

Data flow:

`extension → language registry → grammar parser → SPUR queries → generic capture adapter → nodes/edges → resolver`

HTML `script_element/raw_text` and `style_element/raw_text` remain opaque in this increment. A future embedded-language design can extend that boundary without changing the standalone contracts here.

In [ ]:
flowchart TD
    SPEC["`@spec HTML-CSS-ROUTING
@type InputKind = enum[html_ext, htm_ext, css_ext, existing_ext, unsupported_ext, embedded_script_raw, embedded_style_raw]
@type Status = enum[html, css, existing, unsupported, opaque]
@input input: InputKind
@output status: Status
@requires VALID: true`"]
    HTML["`@branch HTML
@when input = html_ext or input = htm_ext
@ensures HTML_ROUTE: status = html`"]
    CSS["`@branch CSS
@when input = css_ext
@ensures CSS_ROUTE: status = css`"]
    EXISTING["`@branch EXISTING
@when input = existing_ext
@ensures EXISTING_ROUTE: status = existing`"]
    UNSUPPORTED["`@branch UNSUPPORTED
@when input = unsupported_ext
@ensures UNSUPPORTED_ROUTE: status = unsupported`"]
    OPAQUE["`@branch OPAQUE
@when input = embedded_script_raw or input = embedded_style_raw
@ensures OPAQUE_ROUTE: status = opaque`"]
    CHECK["`@verify CONSISTENT: witness consistency
@verify DETERMINISTIC: prove determinism
@verify COVERED: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> HTML --> CHECK
    SPEC --> CSS --> CHECK
    SPEC --> EXISTING --> CHECK
    SPEC --> UNSUPPORTED --> CHECK
    SPEC --> OPAQUE --> CHECK

## Symbol and relation contract

| Language | Source construct | Emission |
|---|---|---|
| HTML | Element with an `id` attribute | `NodeKind::Section`, label is the quote-free attribute value |
| HTML | `<script src>` | `RelationKind::Imports` |
| HTML | `<link rel="stylesheet" href>` | `RelationKind::Imports` |
| HTML | `<a href>` | `RelationKind::Links` |
| HTML | `img/source/audio/video` source attributes and video poster | `RelationKind::Links` |
| CSS | `rule_set` | `NodeKind::Section`, label is selector source text trimmed at its boundaries |
| CSS | `@keyframes` | `NodeKind::Function`, label is the keyframe name |
| CSS | declaration whose property begins `--` | `NodeKind::Constant`, label is the custom property name |
| CSS | `@import` target | `RelationKind::Imports` |
| CSS | `url(...)` target | `RelationKind::Links` |

Selector labels preserve internal source whitespace; v1 does not implement semantic selector canonicalization. Multiple definitions with the same label remain distinct by file/range identity.

Existing generic behavior supplies `Contains` and `Defines`. Dependency edges originate from the nearest extracted containing symbol, falling back to the file node when no ID-bearing HTML region or CSS rule contains the reference. Failed resolution preserves the normalized target label as evidence.

Explicitly non-emitting syntax includes ordinary HTML tags, `class` attributes, inline style attributes, ordinary CSS properties, selector fragments as separate symbols, and embedded script/style definitions.

## Failure and compatibility behavior

- Grammar setup or tree-sitter query compilation failure aborts the affected language group. Broken extraction must not silently degrade to an empty graph.
- A file read failure, invalid UTF-8 source, or per-file extraction failure follows existing batch behavior: emit a warning, skip that file, and continue.
- Tree-sitter may return a partial tree for malformed HTML/CSS. Queries may extract valid regions, but the extractor must not panic or synthesize captures unsupported by the tree.
- HTML grammar compatibility with the workspace's `tree-sitter 0.25.10` is a release gate because `tree-sitter-html 0.23.2` was tested upstream with a 0.24 development dependency.
- CSS and HTML edge values must capture quote-free inner values where the grammar exposes them. No network access or filesystem target validation occurs during extraction.

## Test and acceptance strategy

Tests must establish:

1. case-insensitive routing for `.html`, `.htm`, and `.css`;
2. grammar loading and compilation of every owned query;
3. each approved node kind, label, byte range, enclosing scope, and automatic containment/definition edge;
4. each approved import/link relation, including unresolved target-label evidence;
5. no symbols for ordinary tags, class attributes, ordinary CSS properties, or embedded script/style definitions;
6. no routing for SCSS/Less/template extensions;
7. malformed-input tolerance without panic;
8. unique registry extensions and complete definition/relation gate rows;
9. synchronized human-readable coverage matrices;
10. no regressions under `scripts/spur-cargo test -p spur-graph`.

Focused fixtures should keep HTML and CSS source, expected nodes, expected edges, and negative assertions close together. Existing-language regression tests remain authoritative; the feature must not update unrelated snapshots merely to obtain green tests.

In [ ]:
flowchart TD
    SPEC["`@spec HTML-CSS-RELEASE
@type Status = enum[eligible, blocked]
@input grammar_compatible: Bool
@input registry_complete: Bool
@input queries_compile: Bool
@input positive_fixtures: Bool
@input negative_fixtures: Bool
@input coverage_synced: Bool
@input regression_green: Bool
@output status: Status
@requires VALID: true`"]
    ELIGIBLE["`@branch ELIGIBLE
@when grammar_compatible and registry_complete and queries_compile and positive_fixtures and negative_fixtures and coverage_synced and regression_green
@ensures RELEASE: status = eligible`"]
    BLOCKED["`@branch BLOCKED
@when not grammar_compatible or not registry_complete or not queries_compile or not positive_fixtures or not negative_fixtures or not coverage_synced or not regression_green
@ensures HOLD: status = blocked`"]
    CHECK["`@verify CONSISTENT: witness consistency
@verify DETERMINISTIC: prove determinism
@verify COVERED: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> ELIGIBLE --> CHECK
    SPEC --> BLOCKED --> CHECK

## Implementation boundaries and dependency order

The later implementation plan should preserve these boundaries:

1. **Shared RED contract and scaffolding:** add focused failing routing/query/integration tests and the minimal shared dependency/registry structure needed to compile those tests.
2. **HTML extraction:** own HTML query files and HTML-specific fixtures.
3. **CSS extraction:** own CSS query files and CSS-specific fixtures. HTML and CSS query work can proceed in parallel after shared scaffolding because their files and grammar contracts are independent.
4. **Integration and contract synchronization:** complete shared `languages.rs` gates, coverage matrices, full crate verification, and documentation after both language paths are green.

Workers for HTML and CSS must not concurrently edit the same shared registry/test-contract sections. The implementation plan should isolate those shared edits into predecessor/successor tasks rather than accepting merge conflict risk.

## Risks, migration, and future work

### Risks

- HTML grammar/runtime ABI incompatibility: mitigated by an explicit compile-and-parse gate.
- Query over-capture: mitigated by positive and negative fixtures and the low-noise ontology.
- Selector-label instability under formatting: accepted for v1; stable IDs already include symbol identity and range, and no canonicalizer is introduced.
- Excess link noise from `url(...)`: mitigate with fixtures covering data URLs, fragments, and malformed calls; exclude values that cannot produce a meaningful target label.
- Coverage drift: prevented by executable registry/query gates plus README matrix synchronization.

### Migration

This is additive language support. No serialized enum variant is added to `NodeKind` or `RelationKind`, so no graph schema or artifact-version migration is required. Existing cached graphs gain HTML/CSS facts only after their normal rebuild/freshness path sees the new extractor version.

### Deferred follow-ups

Embedded JavaScript/CSS extraction, selector canonicalization, cross-file class/id binding, DOM/cascade semantics, and template-language support require separate workflow-backed designs.

## Review checklist

The written specification is acceptable when:

- every approved conversational decision appears here without contradiction;
- there are no unresolved placeholders;
- both formal cells execute with fresh proof evidence;
- routing is total and exclusive over the declared finite input categories;
- release eligibility is equivalent to every mandatory gate being true;
- scope explicitly excludes embedded-language recursion and ontology expansion;
- the design epic records the notebook path, formal spec IDs, and proof hashes.

After written-spec approval, close design epic `bd-rnwv` and transition only to the `writing-plans` workflow. Do not implement directly from this notebook without the beads-backed plan.